In [16]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ORCL', 'IBM', 'CRM']
ENGINEERED_DIR = "data/engineered"
SENTIMENT_PATH = "data/news/daily_sentiment_summary_et.csv"
PROCESSED_DIR = "data/processed"
ML_READY_DIR = "data/ml_ready"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ML_READY_DIR, exist_ok=True)

# Join sentiment and technical features into a unified dataset
sent_df = pd.read_csv(SENTIMENT_PATH)
sent_df['date'] = pd.to_datetime(sent_df['date'])

for ticker in TICKERS:
    tech_path = f"{ENGINEERED_DIR}/{ticker}_engineered.csv"
    if not os.path.exists(tech_path):
        print(f"[Skip] Price file not found: {tech_path}")
        continue

    tmp_df = pd.read_csv(tech_path, nrows=1)
    date_col = "date" if "date" in tmp_df.columns else "Date"
    tech_df = pd.read_csv(tech_path, parse_dates=[date_col])
    tech_df = tech_df.rename(columns={date_col: "date"})
    merged = pd.merge(
        tech_df,
        sent_df[sent_df['ticker'] == ticker],
        on="date",
        how="inner"
    ).sort_values("date").reset_index(drop=True)

    if merged.empty:
        print(f"[Skip] No merged data for {ticker}")
        continue
    merged.to_csv(f"{PROCESSED_DIR}/{ticker}_merged.csv", index=False)
    print(f"[OK] Saved merged file: {PROCESSED_DIR}/{ticker}_merged.csv")

# Shift target variable appropriately (next-day prediction)
for ticker in TICKERS:
    m_path = f"{PROCESSED_DIR}/{ticker}_merged.csv"
    if not os.path.exists(m_path):
        continue
    df = pd.read_csv(m_path, parse_dates=['date'])
    if 'Close' not in df.columns:
        print(f"[Skip] {ticker}: No 'Close' column")
        continue
    df['target_close'] = df['Close'].shift(-1)
    df = df.dropna(subset=['target_close']).reset_index(drop=True)
    df.to_csv(m_path, index=False)
    print(f"[OK] Added target column and saved: {m_path}")

# Create training/validation/test split.
def chronological_split(df, test_size=0.25):
    """Returns train, test DataFrames split chronologically"""
    n = len(df)
    n_train = int(np.floor((1-test_size) * n))
    train = df.iloc[:n_train]
    test = df.iloc[n_train:]
    return train, test

for ticker in TICKERS:
    m_path = f"{PROCESSED_DIR}/{ticker}_merged.csv"
    if not os.path.exists(m_path):
        continue
    df = pd.read_csv(m_path, parse_dates=['date'])
    exclude = ["date", "ticker", "target_close"]
    feature_cols = [col for col in df.columns if col not in exclude]
    feature_cols = [col for col in feature_cols if df[col].dtype != 'O']
    if len(df) < 5:
        print(f"[Skip] {ticker}: Not enough samples")
        continue
    train_df, test_df = chronological_split(df, test_size=0.25)
    if train_df.empty or test_df.empty:
        print(f"[Skip] {ticker}: Empty split")
        continue
# Scale features using Min-Max or StandardScaler, I choose StandardScaler
    scaler = StandardScaler() 
    X_train = scaler.fit_transform(train_df[feature_cols])
    X_test  = scaler.transform(test_df[feature_cols])
    y_train = train_df['target_close'].values
    y_test  = test_df['target_close'].values

# Milestones: Final ML dataset created and saved. Ready-to-train data format: X_train, y_train, etc.
    out_dir = f"{ML_READY_DIR}/{ticker}"
    os.makedirs(out_dir, exist_ok=True)
    np.save(f"{out_dir}/X_train.npy", X_train)
    np.save(f"{out_dir}/X_test.npy", X_test)
    np.save(f"{out_dir}/y_train.npy", y_train)
    np.save(f"{out_dir}/y_test.npy", y_test)

    import joblib
    joblib.dump(scaler, f"{out_dir}/scaler.joblib")
    print(f"[OK] Saved ML-ready splits for {ticker} ({len(X_train)} train, {len(X_test)} test)")

print("All eligible tickers processed! ML datasets are ready.")

[OK] Saved merged file: data/processed/AAPL_merged.csv
[OK] Saved merged file: data/processed/MSFT_merged.csv
[OK] Saved merged file: data/processed/GOOGL_merged.csv
[OK] Saved merged file: data/processed/AMZN_merged.csv
[OK] Saved merged file: data/processed/META_merged.csv
[OK] Saved merged file: data/processed/NVDA_merged.csv
[OK] Saved merged file: data/processed/TSLA_merged.csv
[OK] Saved merged file: data/processed/ORCL_merged.csv
[OK] Saved merged file: data/processed/IBM_merged.csv
[OK] Saved merged file: data/processed/CRM_merged.csv
[OK] Added target column and saved: data/processed/AAPL_merged.csv
[OK] Added target column and saved: data/processed/MSFT_merged.csv
[OK] Added target column and saved: data/processed/GOOGL_merged.csv
[OK] Added target column and saved: data/processed/AMZN_merged.csv
[OK] Added target column and saved: data/processed/META_merged.csv
[OK] Added target column and saved: data/processed/NVDA_merged.csv
[OK] Added target column and saved: data/process

In [18]:
# Load + inspect all saved splits
import numpy as np
import joblib
import os

TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ORCL', 'IBM', 'CRM']

for ticker in TICKERS:
    ml_dir = f"data/ml_ready/{ticker}"
    print("="*60)
    print(f"{ticker} ML Data:")
    try:
        X_train = np.load(os.path.join(ml_dir, "X_train.npy"))
        X_test = np.load(os.path.join(ml_dir, "X_test.npy"))
        y_train = np.load(os.path.join(ml_dir, "y_train.npy"))
        y_test = np.load(os.path.join(ml_dir, "y_test.npy"))
        scaler = joblib.load(os.path.join(ml_dir, "scaler.joblib"))

        print(f"  X_train shape: {X_train.shape}")
        print(f"  X_test shape:  {X_test.shape}")
        print(f"  y_train shape: {y_train.shape}")
        print(f"  y_test shape:  {y_test.shape}")
        print("  First 2 rows of X_train:\n", X_train[:2])
        print("  First 2 y_train values:", y_train[:2])
        print("  Scaler mean (first 5 features):", scaler.mean_[:5])
        print("  Scaler scale (first 5 features):", scaler.scale_[:5])
    except Exception as e:
        print(f"  [Error] Could not load data for {ticker}: {e}")

print("="*60)

AAPL ML Data:
  X_train shape: (6, 36)
  X_test shape:  (2, 36)
  y_train shape: (6,)
  y_test shape:  (2,)
  First 2 rows of X_train:
 [[-1.46231245e+00 -1.27255873e+00 -1.41094061e+00 -1.25451097e+00
  -1.17108262e+00  4.27791435e-02 -1.91376745e+00 -2.99249397e-01
   1.95062411e+00 -8.86556371e-01  2.47375171e-01 -4.35898061e-01
  -1.22065582e+00 -6.43076373e-01  3.12294217e-01  2.22790813e+00
  -1.15133670e+00 -1.78004571e+00 -4.43918031e-01  1.28593957e+00
   1.18809362e+00 -2.23606798e+00  3.27506295e-01  4.27791435e-02
   3.12294217e-01 -1.19620069e+00 -1.30836580e+00  1.40814071e+00
   1.12157132e-01 -7.07106781e-01 -6.31364150e-01  3.44880749e-01
   1.05999788e-01 -5.90849871e-01  4.25777757e-01  1.76855081e-01]
 [ 9.99991656e-02 -1.50365711e-01 -1.05919746e-01 -3.35806229e-01
   1.10384801e-04 -7.56908777e-01  3.45979725e-02 -6.14267756e-01
  -1.43244434e-01 -5.02550087e-01 -6.63557104e-01  5.67288054e-01
  -5.15386008e-01 -7.74811989e-01 -6.31543906e-01 -3.69203561e-01
  -9.